# Trapping an ion: the Paul trap

**Professor Nano Labs · Quantum computing**

**Try it live in your browser:** [Paul trap lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/paul-trap/)
**Watch first:** [How to trap ions for quantum computing?](https://www.youtube.com/watch?v=8p6P-QTQM2M)
**Watch later:** [How exactly does trapped ions perform quantum computing?](https://www.youtube.com/watch?v=5qpXdGSLWig) · [The intuition behind Gauss divergence theorem](https://www.youtube.com/watch?v=QR5_Nw_pqGc) · [Solving the quantum harmonic oscillator with ladder operators](https://www.youtube.com/watch?v=jHjebJtBXUs)

A $^{40}$Ca$^+$ ion sits in a linear Paul trap: four rods, each $r_0=0.5$ mm from the axis, with a voltage $U_{\rm DC}-V\cos\Omega_{\rm RF}t$ between the two pairs. Near the axis its potential energy is
$$U(x,y,t)=\frac{e}{2r_0^2}\big(U_{\rm DC}-V\cos\Omega_{\rm RF}t\big)(x^2-y^2).$$

The four sections match the lab's four tabs: why no static field can trap (Earnshaw), how the oscillating saddle does, the Mathieu stability chart, and ion crystals with their normal modes. It needs only NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
e, amu, eps0, kB = 1.602176634e-19, 1.66053906660e-27, 8.8541878128e-12, 1.380649e-23
m = 39.9626 * amu      # 40Ca+
r0 = 0.5e-3            # electrode distance (m)

def q_of(V, f):
    # RF strength for amplitude V (volts) at frequency f (Hz)
    return 2 * e * V / (m * r0**2 * (2 * np.pi * f)**2)

def a_of(U_dc, f):
    # steady strength for a DC voltage U_dc (volts)
    return 4 * e * U_dc / (m * r0**2 * (2 * np.pi * f)**2)

print(f'q at 300 V, 20 MHz: {q_of(300, 20e6):.4f}')

## 1 · No static trap: Earnshaw's theorem

In empty space $\nabla^2\Phi=0$. Near the centre the potential is quadratic, $U=\tfrac12 m\omega_0^2(k_xx^2+k_yy^2+k_zz^2)$, so Laplace's equation says $k_x+k_y+k_z=0$. Any quadratic potential can be rotated to this form, and its curvatures are the eigenvalues of a traceless matrix: at least one is negative, unless all are zero. Check it with random potentials, then follow an ion released on a saddle.

In [ ]:
rng = np.random.default_rng(1)
worst = np.inf
for _ in range(10000):
    M = rng.normal(size=(3, 3)); M = (M + M.T) / 2
    M -= np.trace(M) / 3 * np.eye(3)          # Laplace: the curvatures sum to zero
    k = np.linalg.eigvalsh(M)
    worst = min(worst, -k[0])                  # how negative the most negative curvature is
print('every random potential has a downhill direction:', worst > 0)

# the ion released from rest at 0.12 r0 along each axis, time in units of 1/omega0
u0, t = 0.12, np.linspace(0, 12, 1201)
def motion(k, t):
    return u0 * (np.cos(np.sqrt(k) * t) if k > 0 else np.cosh(np.sqrt(-k) * t) if k < 0 else np.ones_like(t))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (kx, ky) in zip(axs, [(1, -1), (0.5, 0.5)]):
    kz = -(kx + ky)
    for lab, k in zip('xyz', (kx, ky, kz)):
        u = motion(k, t)
        out = np.abs(u) >= 1
        if out.any(): u[np.argmax(out) + 1:] = np.nan       # stop at the electrodes
        ax.plot(t, u, label=f'{lab}: k = {k:+.2f}')
    ax.axhspan(1, 1.1, color='r', alpha=.1); ax.axhspan(-1.1, -1, color='r', alpha=.1)
    ax.set(xlabel='time (1/ω0)', ylabel='position (r0)', ylim=(-1.1, 1.1), title=f'kx = {kx}, ky = {ky}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('escape time for k = -1:', np.arccosh(1 / u0))

## 2 · Flip the saddle

With the RF on and $U_{\rm DC}=0$, measuring time as $\xi=\Omega_{\rm RF}t/2$ turns the motion along $x$ into the Mathieu equation
$$\frac{d^2x}{d\xi^2}+(a-2q\cos2\xi)\,x=0,\qquad q=\frac{2eV}{m r_0^2\Omega_{\rm RF}^2},$$
and along $y$ the same with $(a,q)\to(-a,-q)$. For small $q$ the ion moves as $x\approx x_1\cos\omega t\,[1-\tfrac q2\cos\Omega_{\rm RF}t]$: a slow swing at $\omega\approx q\Omega_{\rm RF}/2\sqrt2$, the secular motion, with a small jiggle on top, the micromotion. Integrate the exact equation with fourth-order Runge–Kutta and compare.

In [ ]:
def mathieu(a, q, u0, cycles, per=120):
    # integrate u'' = -(a - 2q cos 2 xi) u from rest; one RF period is xi = pi
    h = np.pi / per; n = int(cycles * per)
    f = lambda xi, u: -(a - 2 * q * np.cos(2 * xi)) * u
    u, v, out = u0, 0.0, [u0]
    for i in range(n):
        xi = i * h
        k1u, k1v = v, f(xi, u)
        k2u, k2v = v + h/2 * k1v, f(xi + h/2, u + h/2 * k1u)
        k3u, k3v = v + h/2 * k2v, f(xi + h/2, u + h/2 * k2u)
        k4u, k4v = v + h * k3v, f(xi + h, u + h * k3u)
        u, v = u + h/6 * (k1u + 2*k2u + 2*k3u + k4u), v + h/6 * (k1v + 2*k2v + 2*k3v + k4v)
        out.append(u)
    return np.arange(n + 1) / per, np.array(out)

V, f = 300, 20e6
q = q_of(V, f)
t, x = mathieu(0, q, 0.3, 25)
x1, b_app = 0.3 / (1 - q / 2), q / np.sqrt(2)
plt.figure(figsize=(9, 3.2))
plt.plot(t, x, lw=1.4, label='exact (Mathieu)')
plt.plot(t, x1 * np.cos(b_app * np.pi * t), '--', label='slow swing, bowl picture')
plt.xlabel('time (RF periods)'); plt.ylabel('x / r0'); plt.legend(); plt.title(f'q = {q:.3f}'); plt.show()
print(f'bowl-picture secular frequency: {q * f / (2 * np.sqrt(2)) / 1e6:.3f} MHz')
print(f'trap depth (bowl picture, qV/8): {q * V / 8:.2f} eV  ~ {q * V / 8 * e / kB:.3g} K')

## 3 · The stability chart

Over one RF period the motion is a linear map, $(u,u')\mapsto M(u,u')$ with $\det M=1$. The motion stays bounded when $|\operatorname{tr}M|<2$, and then $\cos\pi\beta=\tfrac12\operatorname{tr}M$ gives the secular frequency $\omega=\beta\,\Omega_{\rm RF}/2$. Compute the trace on a grid of $(a,q)$, all at once. The ion is trapped where both $(a,q)$ (along $x$) and $(-a,-q)$ (along $y$) are stable.

In [ ]:
def trace(a, q, n=240):
    # trace of the one-period transfer matrix; a and q can be arrays
    a, q = np.broadcast_arrays(np.asarray(a, float), np.asarray(q, float))
    h = np.pi / n
    S = [np.ones_like(a), np.zeros_like(a), np.zeros_like(a), np.ones_like(a)]   # u1, v1, u2, v2
    def F(xi, S):
        w = -(a - 2 * q * np.cos(2 * xi))
        return [S[1], w * S[0], S[3], w * S[2]]
    for i in range(n):
        xi = i * h
        k1 = F(xi, S)
        k2 = F(xi + h/2, [s + h/2 * k for s, k in zip(S, k1)])
        k3 = F(xi + h/2, [s + h/2 * k for s, k in zip(S, k2)])
        k4 = F(xi + h, [s + h * k for s, k in zip(S, k3)])
        S = [s + h/6 * (p + 2*r + 2*w + z) for s, p, r, w, z in zip(S, k1, k2, k3, k4)]
    return S[0] + S[3]

def beta(a, q):
    tr = trace(a, q)
    return np.where(np.abs(tr) < 2, np.arccos(np.clip(tr / 2, -1, 1)) / np.pi, np.nan)

qq, aa = np.meshgrid(np.linspace(0, 1.2, 241), np.linspace(-0.4, 0.4, 161))
sx, sy = np.abs(trace(aa, qq)) < 2, np.abs(trace(-aa, -qq)) < 2
plt.figure(figsize=(8, 4.2))
plt.contourf(qq, aa, sx.astype(float), levels=[0.5, 1.5], colors=['#c47a12'], alpha=.25)
plt.contourf(qq, aa, sy.astype(float), levels=[0.5, 1.5], colors=['#1f6fb4'], alpha=.25)
plt.contourf(qq, aa, (sx & sy).astype(float), levels=[0.5, 1.5], colors=['#a3326c'], alpha=.6)
plt.axvline(0.908, ls=':', c='k')
plt.xlabel('q'); plt.ylabel('a'); plt.title('orange: x stable, blue: y stable, dark: trapped'); plt.show()

# the a = 0 edge, by bisection on the trace
lo, hi = 0.8, 1.0
for _ in range(50):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if abs(trace(0, mid)) < 2 else (lo, mid)
print('edge of stability at a = 0: q =', round(lo, 5))
qs = np.linspace(0.01, 0.9, 90)
plt.figure(figsize=(6, 3.2))
plt.plot(qs, beta(0, qs), label='exact'); plt.plot(qs, qs / np.sqrt(2), '--', label='bowl formula q/√2')
plt.xlabel('q'); plt.ylabel('β'); plt.legend(); plt.show()

## 4 · Ion crystals and their normal modes

Along the axis the end caps make a bowl at $\omega_z$; across it the RF makes a stiffer one at $\omega_r$. Measuring lengths in $\ell=(e^2/4\pi\varepsilon_0 m\omega_z^2)^{1/3}$, the energy of $N$ ions in the $x$–$z$ plane is
$$\frac{U}{m\omega_z^2\ell^2}=\sum_i\tfrac12\big(z_i^2+b^2x_i^2\big)+\sum_{i<j}\frac1{r_{ij}},\qquad b=\omega_r/\omega_z.$$
Find the minimum with Newton's method, then diagonalize the second derivatives: their eigenvalues are $(\omega/\omega_z)^2$ for the normal modes.

In [ ]:
def energy(P, N, b):
    z, x = P[:N], P[N:]
    dz, dx = z[:, None] - z, x[:, None] - x
    r = np.hypot(dz, dx) + np.eye(N)
    return 0.5 * np.sum(z**2 + b * b * x**2) + 0.5 * np.sum((1 / r)[~np.eye(N, dtype=bool)])

def gradient(P, N, b):
    z, x = P[:N], P[N:]
    dz, dx = z[:, None] - z, x[:, None] - x
    r3 = (np.hypot(dz, dx) + np.eye(N))**3
    return np.concatenate([z - np.sum(dz / r3, axis=1), b * b * x - np.sum(dx / r3, axis=1)])

def hessian(P, N, b):
    H = np.zeros((2 * N, 2 * N)); H[:N, :N] = np.eye(N); H[N:, N:] = b * b * np.eye(N)
    for i in range(N):
        for j in range(N):
            if i == j: continue
            d = np.array([P[i] - P[j], P[N + i] - P[N + j]]); r = np.linalg.norm(d)
            K = (3 * np.outer(d, d) - r * r * np.eye(2)) / r**5      # second derivatives of 1/r
            for s in range(2):
                for t in range(2):
                    H[s * N + i, t * N + i] += K[s, t]; H[s * N + i, t * N + j] -= K[s, t]
    return H

def crystal(N, b):
    # start from a slightly zigzagged string; saddle-free Newton steps walk downhill even from a saddle
    P = np.concatenate([(np.arange(N) - (N - 1) / 2) * 1.6 * N**-0.45, 0.02 * (-1.0)**np.arange(N)])
    for _ in range(300):
        g = gradient(P, N, b)
        if np.linalg.norm(g) < 1e-12: break
        lam, vec = np.linalg.eigh(hessian(P, N, b))
        d = -vec @ ((vec.T @ g) / np.maximum(np.abs(lam), 1e-6))
        s, E0 = 1.0, energy(P, N, b)
        while energy(P + s * d, N, b) > E0 and s > 1e-9: s /= 2
        P = P + s * d
    w = np.sqrt(np.clip(np.linalg.eigvalsh(hessian(P, N, b)), 0, None))
    return P[:N], P[N:], w

fz = 1e6
ell = (e**2 / (4 * np.pi * eps0 * m * (2 * np.pi * fz)**2))**(1 / 3)
print(f'length scale l = {ell * 1e6:.3f} um at wz/2pi = 1 MHz')
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for N in range(1, 11):
    z, x, w = crystal(N, 6.0)                       # strong radial grip: always a straight string
    axs[0].plot(z * ell * 1e6, np.full(N, N), 'o', c='#a3326c')
    axs[1].plot(np.full(N, N), np.sqrt(np.linalg.eigvalsh(hessian(np.concatenate([z, x]), N, 6.0)[:N, :N])), 'o', c='#a3326c')
axs[0].set(xlabel='position along the axis (um)', ylabel='N', title='where the ions sit')
axs[1].axhline(1, ls='--', c='gray'); axs[1].axhline(np.sqrt(3), ls='--', c='gray')
axs[1].set(xlabel='N', ylabel='ω / ωz', title='axial modes: COM at 1, breathing at √3')
plt.tight_layout(); plt.show()

# the zigzag transition: the string stays straight while b exceeds sqrt((mu_max - 1) / 2)
for N in (3, 5, 10):
    z, x, w = crystal(N, 10.0)
    mu = np.linalg.eigvalsh(hessian(np.concatenate([z, x]), N, 0.0)[:N, :N])[-1]
    print(f'N = {N:2d}: straight while wr/wz > {np.sqrt((mu - 1) / 2):.4f}')
z, x, w = crystal(10, 4.0)
print('ten ions at wr/wz = 4: largest sideways offset', np.abs(x).max().round(3), '(a zigzag)')

## Check the lab's answers

These asserts confirm the numbers quoted in the lab's "Predict first" answers.

In [ ]:
# Q1: a bowl in x-y forces kz = -1, and the ion escapes along z at t = 2.81 / w0
assert np.isclose(np.arccosh(1 / 0.12), 2.81, atol=0.005)
# Q2: 300 V at 5 MHz gives q = 5.9, far beyond 0.908; stability needs f above 12.7 MHz
assert np.isclose(q_of(300, 5e6), 5.87, atol=0.01) and abs(trace(0, q_of(300, 5e6))) > 2
assert np.isclose(np.sqrt(2 * e * 300 / (m * r0**2 * 0.908046)) / (2 * np.pi) / 1e6, 12.71, atol=0.01)
# Q3: q = 0.367, beta = 0.267, 7.5 RF periods per swing, 2.67 MHz (2.59 MHz from the bowl formula)
q3 = q_of(300, 20e6); b3 = float(beta(0, q3))
assert np.isclose(b3, 0.2669, atol=5e-4) and np.isclose(2 / b3, 7.49, atol=0.01) and np.isclose(q3 * 20 / (2 * np.sqrt(2)), 2.59, atol=0.005)
# Q4: at 40 MHz q falls to a quarter, the secular frequency to 1.30 MHz, the depth from 13.8 to 3.4 eV
q4 = q_of(300, 40e6)
assert np.isclose(q4, q3 / 4) and np.isclose(float(beta(0, q4)) * 20, 1.30, atol=0.005)
assert np.isclose(q3 * 300 / 8, 13.76, atol=0.01) and np.isclose(q4 * 300 / 8, 3.44, atol=0.01)
# Q5: the edge at q = 0.908; beta = 0.916 at q = 0.9 against 0.64 from the bowl formula
assert np.isclose(lo, 0.90805, atol=1e-4) and np.isclose(float(beta(0, 0.9)), 0.916, atol=1e-3)
# Q6: a = 0.1, q = 0.4: x stable (beta 0.443), y lost; y needs |a| < 0.079 at this q
assert np.isclose(float(beta(0.1, 0.4)), 0.4433, atol=5e-4) and np.isnan(beta(-0.1, -0.4))
assert abs(trace(-0.0785, 0.4)) < 2 < abs(trace(-0.0790, 0.4))
# Q7: two ions 2^(1/3) l = 5.6 um apart; COM and breathing modes at 1 and sqrt 3 for every N
z2, x2, w2 = crystal(2, 4.0)
assert np.isclose((z2.max() - z2.min()) * ell * 1e6, 5.605, atol=0.005)
for N in range(2, 11):
    z, x, w = crystal(N, 8.0)
    ax_modes = np.sqrt(np.linalg.eigvalsh(hessian(np.concatenate([z, x]), N, 8.0)[:N, :N]))
    assert np.allclose(ax_modes[:2], [1, np.sqrt(3)], atol=1e-8)
# Q8: ten ions need wr/wz > 4.60 to stay straight
z, x, w = crystal(10, 10.0)
mu = np.linalg.eigvalsh(hessian(np.concatenate([z, x]), 10, 0.0)[:10, :10])[-1]
assert np.isclose(np.sqrt((mu - 1) / 2), 4.596, atol=1e-3)
print('All checks passed.')

## Things to try

Some ways to take the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/paul-trap/) further.

1. **A ring trap.** In a 3D Paul trap the ion feels $(a,q)$ along $z$ and $(-a/2,-q/2)$ across it. Redraw the stability chart. Where is the first stable region now?
2. **A mass filter.** Fix the ratio $a/q$ and vary the mass. Which masses are stable in both directions for a line just below the tip of the trapped region?
3. **Start with the RF at a different phase.** Release the ion at $\xi=\pi/4$ instead of 0. How does the size of the micromotion change, and does the secular frequency?
4. **A three-dimensional crystal.** Add the $y$ coordinate to the crystal energy. For ten ions with weak radial grip, do you get a zigzag or a helix?